# Lesson 07 Lab — From DRAM Cells to HBM Packaging

**Puzzle:** HBM cells are still DRAM, so where does high bandwidth actually come from?

This notebook retains one complete RTX 5090 execution.


## Why this matters

HBM stacks DRAM dies and connects many signals through TSVs and microbumps to a base die and silicon interposer near the GPU package. The cell still stores charge and needs sense amplifiers, rows, banks, and refresh. The bandwidth gain comes primarily from a very wide, highly parallel interface and package integration—not from turning DRAM cells into SRAM.


## 0. Predict before running

1. Trace one read from a cell array to the GPU memory controller.
2. Calculate bandwidth for a 512-bit interface at 28 Gb/s per pin.
3. Predict why a copy benchmark cannot reach the exact theoretical number.

For each prediction, write the observation that would disprove it.


## 1. Theory and mechanism

Theoretical interface bandwidth is `bus_width_bits × pin_rate / 8`. The notebook calculates that relationship and measures a large device-to-device copy on the available RTX 5090, which uses GDDR7 rather than HBM. This contrast is intentional: the equation generalizes, while the recorded GPU truthfully identifies its external-memory technology. Effective copy bandwidth includes both a read and a write in the reported byte accounting.

- HBM is external package memory, not an SM-local cache.
- TSVs and interposers create a wide path; banks provide internal parallelism.
- Theoretical interface bandwidth and achieved application bandwidth are different quantities.


## 2. Trace the mechanism

### Mechanism map

```mermaid
flowchart LR
  A["1T1C arrays"] --> B["sense amps + banks"]
  B --> C["HBM stack + base die"]
  C --> D["TSV / microbump"]
  D --> E["interposer"]
  E --> F["GPU memory controller"]
```


## 3. Inspect the visual boundary

![HBM circuit-to-package path](../assets/HBM_circuit_to_gpu_connection.png)

- [Printable HBM diagram](../assets/HBM_circuit_to_gpu_connection_A4_portrait.pdf)

These are conceptual teaching diagrams. They explain the named data path and are not die-accurate schematics of a particular commercial GPU.


## 4. Inspect the execution environment

The next cell asserts CUDA, records GPU/PyTorch/CUDA identity, fixes the seed, and defines the common event-timing helpers.


In [1]:
LESSON_NO = 7
LESSON_TITLE = 'From DRAM Cells to HBM Packaging'

from pathlib import Path
from collections import Counter, deque
import json, math, platform, statistics, sys, time

import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), "Chapter 04 retained runs require a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260813 + LESSON_NO
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

major, minor = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
ENV = {
    "gpu": torch.cuda.get_device_name(0),
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    pos = (len(ordered) - 1) * q
    lo, hi = math.floor(pos), math.ceil(pos)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def cuda_samples(fn, warmup=5, repeats=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    samples = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True)
        stop = torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        stop.record()
        stop.synchronize()
        samples.append(float(start.elapsed_time(stop)))
    return samples

def summary(samples):
    return {
        "median_ms": statistics.median(samples),
        "p95_ms": percentile(samples, 0.95),
        "samples_ms": samples,
    }


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "seed": 20260820
}


## 5. Freeze the experiment

| Role | Frozen value |
|---|---|
| Baseline | theoretical bandwidth from interface width and pin rate |
| Candidate | RTX 5090 device-copy effective bandwidth |
| Held constant | tensor size, dtype, warm-up, repetitions, and event timing |
| Measurements | theoretical GB/s, copy median, effective GB/s, and achieved/theoretical ratio |
| Evidence | `pytorch-gpu` |

**Experiment:** Calculate width-based bandwidth and measure large CUDA device copies.


## 6. Inspect the code

A preallocated source and destination avoid allocator timing. `copy_` is repeated between CUDA events, and requested traffic counts source read plus destination write. The formula example matches the official 5090 interface fields but does not relabel GDDR7 as HBM.

Do not run until the code matches the frozen table.


In [2]:
interface_bits = 512
pin_rate_gbps = 28.0
theoretical_gbps = interface_bits * pin_rate_gbps / 8
n = 2**26
src = torch.randn(n, device=DEVICE, dtype=torch.float32)
dst = torch.empty_like(src)
samples = cuda_samples(lambda: dst.copy_(src), repeats=25)
median = statistics.median(samples)
requested_bytes = 2 * src.numel() * src.element_size()
effective = requested_bytes / (median / 1e3) / 1e9
metrics = {
    "memory_technology": "GDDR7",
    "interface_bits": interface_bits,
    "pin_rate_gbps": pin_rate_gbps,
    "theoretical_gbps": theoretical_gbps,
    "tensor_mib": src.numel() * src.element_size() / 2**20,
    "copy_median_ms": median,
    "effective_copy_gbps": effective,
    "achieved_fraction": effective / theoretical_gbps,
    "samples_ms": samples,
    "checksum": float(dst[:4096].sum().item()),
}
analysis = (
    f"A 512-bit interface at 28 Gb/s per pin yields {theoretical_gbps:.0f} GB/s. The "
    f"device-copy probe reported {effective:.1f} requested GB/s ({effective/theoretical_gbps:.1%} "
    "of that interface number) on the GDDR7 RTX 5090."
)
print(json.dumps(metrics, indent=2))


{
  "memory_technology": "GDDR7",
  "interface_bits": 512,
  "pin_rate_gbps": 28.0,
  "theoretical_gbps": 1792.0,
  "tensor_mib": 256.0,
  "copy_median_ms": 0.3529599905014038,
  "effective_copy_gbps": 1521.0531687666303,
  "achieved_fraction": 0.8488019914992356,
  "samples_ms": [
    0.36585599184036255,
    0.3553920090198517,
    0.3516480028629303,
    0.3521920144557953,
    0.3533119857311249,
    0.3543359935283661,
    0.3517119884490967,
    0.3550719916820526,
    0.35385599732398987,
    0.35305601358413696,
    0.3526400029659271,
    0.3527680039405823,
    0.35315200686454773,
    0.3564159870147705,
    0.35516801476478577,
    0.3529599905014038,
    0.3537920117378235,
    0.35257598757743835,
    0.35225600004196167,
    0.35280001163482666,
    0.3528960049152374,
    0.3548479974269867,
    0.3518719971179962,
    0.3527359962463379,
    0.35180801153182983
  ],
  "checksum": -15.691978454589844
}


## 7. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Theoretical interface bandwidth | 1,792.0000 |
| Copy median | 0.353 ms |
| Effective copy bandwidth | 1,521.0532 |
| Achieved/theoretical | 84.88% |


## 8. Explain rather than overclaim

A 512-bit interface at 28 Gb/s per pin yields 1792 GB/s. The device-copy probe reported 1521.1 requested GB/s (84.9% of that interface number) on the GDDR7 RTX 5090.

**Evidence boundary:** CUDA work executed through PyTorch. It does not identify an internal instruction, cache event, or proprietary hardware block without additional profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, metrics, analysis, evidence label, and bounded conclusion, then prints the exact JSON.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 7, "title": 'From DRAM Cells to HBM Packaging', "environment": ENV,
    "evidence_label": 'pytorch-gpu', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Use the wide-interface equation for a ceiling and a controlled benchmark for achieved bandwidth; always name the memory technology and traffic convention.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 7,
  "title": "From DRAM Cells to HBM Packaging",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "seed": 20260820
  },
  "evidence_label": "pytorch-gpu",
  "metrics": {
    "memory_technology": "GDDR7",
    "interface_bits": 512,
    "pin_rate_gbps": 28.0,
    "theoretical_gbps": 1792.0,
    "tensor_mib": 256.0,
    "copy_median_ms": 0.3529599905014038,
    "effective_copy_gbps": 1521.0531687666303,
    "achieved_fraction": 0.8488019914992356,
    "samples_ms": [
      0.36585599184036255,
      0.3553920090198517,
      0.3516480028629303,
      0.3521920144557953,
      0.3533119857311249,
      0.3543359935283661,
      0.3517119884490967,
      0.3550719916820526,
      0.35385599732398987,
      0.35305601358413696,
      0.3526400029659271,
      0.3527680039405823,
      0.35315200686454773,
      0.3564159870147705,
      0.3551680147647

## 10. Make the decision

> Use the wide-interface equation for a ceiling and a controlled benchmark for achieved bandwidth; always name the memory technology and traffic convention.

**Failure analysis:** Copy engines, caches, clocks, thermals, tensor size, ECC, and byte-count conventions affect the ratio. HBM package details differ across products and generations.


## 11. Extend the evidence

Repeat with a streaming triad kernel and profiler DRAM counters, then compare an actual HBM GPU using the identical protocol.

See [`README.md`](README.md) for the full explanation and references.
